<a href="https://colab.research.google.com/github/mtamamulya/PBO/blob/main/JobSheet11_Mukhlish_Pratama_Mulya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***PRAKTIKUM***

In [1]:
# 1. Instalasi library Streamlit dan Pandas
!pip install -q streamlit pandas

# 2. Instalasi localtunnel untuk mengekspos server lokal Colab ke internet publik
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 51.7 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 3s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹

In [2]:
%%writefile konfigurasi.py
# konfigurasi.py
import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
NAMA_DB = 'pengeluaran_harian.db'
DB_PATH = os.path.join(BASE_DIR, NAMA_DB)

KATEGORI_PENGELUARAN = ["Makanan", "Transportasi", "Hiburan", "Tagihan", "Belanja", "Kesehatan", "Pendidikan", "Lainnya"]
KATEGORI_DEFAULT = "Lainnya"

Writing konfigurasi.py


In [3]:
%%writefile database.py
# database.py
import sqlite3
import pandas as pd
from konfigurasi import DB_PATH

def get_db_connection() -> sqlite3.Connection | None:
    """Membuka dan mengembalikan koneksi baru ke database SQLite."""
    try:
        conn = sqlite3.connect(DB_PATH, timeout=10, detect_types=sqlite3.PARSE_DECLTYPES)
        conn.row_factory = sqlite3.Row  # Akses kolom berdasarkan nama
        return conn
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Koneksi DB gagal: {e}")
        return None

def execute_query(query: str, params: tuple = None):
    """Menjalankan query non-SELECT. Mengembalikan lastrowid jika INSERT."""
    conn = get_db_connection()
    if not conn: return None
    last_id = None
    try:
        cursor = conn.cursor()
        if params: cursor.execute(query, params)
        else: cursor.execute(query)
        conn.commit()
        last_id = cursor.lastrowid
        return last_id
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Query gagal: {e}")
        conn.rollback()
        return None
    finally:
        if conn: conn.close()

def fetch_query(query: str, params: tuple = None, fetch_all: bool = True):
    """Menjalankan query SELECT dan mengembalikan hasil."""
    conn = get_db_connection()
    if not conn: return None
    try:
        cursor = conn.cursor()
        if params: cursor.execute(query, params)
        else: cursor.execute(query)
        result = cursor.fetchall() if fetch_all else cursor.fetchone()
        return result
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Fetch gagal: {e}")
        return None
    finally:
        if conn: conn.close()

def get_dataframe(query: str, params: tuple = None) -> pd.DataFrame:
    """Menjalankan query SELECT dan mengembalikan DataFrame Pandas."""
    conn = get_db_connection()
    if not conn: return pd.DataFrame()
    try:
        df = pd.read_sql_query(query, conn, params=params)
        return df
    except Exception as e:
        print(f"ERROR [database.py] Gagal baca ke DataFrame: {e}")
        return pd.DataFrame()
    finally:
        if conn: conn.close()

def setup_database_initial():
    """Memastikan tabel transaksi ada di database."""
    conn = get_db_connection()
    if not conn: return False
    try:
        cursor = conn.cursor()
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK(jumlah > 0),
            kategori TEXT,
            tanggal DATE NOT NULL
        );"""
        cursor.execute(sql_create_table)
        conn.commit()
        return True
    except sqlite3.Error as e:
        return False
    finally:
        if conn: conn.close()

Writing database.py


In [4]:
%%writefile model.py
# model.py
import datetime

class Transaksi:
    """Merepresentasikan satu entitas transaksi pengeluaran (Data Class)."""
    def __init__(self, deskripsi: str, jumlah: float, kategori: str, tanggal: datetime.date | str, id_transaksi: int | None = None):
        self.id = id_transaksi
        self.deskripsi = str(deskripsi) if deskripsi else "Tanpa Deskripsi"

        try:
            jumlah_float = float(jumlah)
            self.jumlah = jumlah_float if jumlah_float > 0 else 0.0
        except:
            self.jumlah = 0.0

        self.kategori = str(kategori) if kategori else "Lainnya"

        if isinstance(tanggal, datetime.date):
            self.tanggal = tanggal
        elif isinstance(tanggal, str):
            try: self.tanggal = datetime.datetime.strptime(tanggal, "%Y-%m-%d").date()
            except: self.tanggal = datetime.date.today()
        else:
            self.tanggal = datetime.date.today()

Writing model.py


In [5]:
%%writefile manajer_anggaran.py
# manajer_anggaran.py
import datetime
import pandas as pd
from model import Transaksi
import database

class AnggaranHarian:
    """Mengelola logika bisnis pengeluaran harian (Repository Pattern)."""
    _db_setup_done = False

    def __init__(self):
        if not AnggaranHarian._db_setup_done:
            if database.setup_database_initial():
                AnggaranHarian._db_setup_done = True

    def tambah_transaksi(self, transaksi: Transaksi) -> bool:
        if not isinstance(transaksi, Transaksi) or transaksi.jumlah <= 0: return False
        sql = "INSERT INTO transaksi (deskripsi, jumlah, kategori, tanggal) VALUES (?, ?, ?, ?)"
        params = (transaksi.deskripsi, transaksi.jumlah, transaksi.kategori, transaksi.tanggal.strftime("%Y-%m-%d"))
        last_id = database.execute_query(sql, params)
        if last_id is not None:
            transaksi.id = last_id
            return True
        return False

    def get_dataframe_transaksi(self, filter_tanggal: datetime.date | None = None) -> pd.DataFrame:
        query = "SELECT tanggal, kategori, deskripsi, jumlah FROM transaksi"
        params = None
        if filter_tanggal:
            query += " WHERE tanggal = ?"
            params = (filter_tanggal.strftime("%Y-%m-%d"),)
        query += " ORDER BY tanggal DESC, id DESC"

        df = database.get_dataframe(query, params=params)
        if not df.empty:
            df.columns = ['Tanggal', 'Kategori', 'Deskripsi', 'jumlah_raw']
            df['Jumlah (Rp)'] = df['jumlah_raw'].map(lambda x: f"Rp {x or 0:,.0f}".replace(",", "."))
            return df[['Tanggal', 'Kategori', 'Deskripsi', 'Jumlah (Rp)']]
        return df

    def hitung_total_pengeluaran(self, tanggal: datetime.date | None = None) -> float:
        sql = "SELECT SUM(jumlah) FROM transaksi"
        params = None
        if tanggal:
            sql += " WHERE tanggal = ?"
            params = (tanggal.strftime("%Y-%m-%d"),)
        result = database.fetch_query(sql, params=params, fetch_all=False)
        if result and result[0] is not None: return float(result[0])
        return 0.0

    def get_pengeluaran_per_kategori(self, tanggal: datetime.date | None = None) -> dict:
        hasil = {}
        sql = "SELECT kategori, SUM(jumlah) FROM transaksi"
        params = []
        if tanggal:
            sql += " WHERE tanggal = ?"
            params.append(tanggal.strftime("%Y-%m-%d"))
        sql += " GROUP BY kategori HAVING SUM(jumlah) > 0 ORDER BY SUM(jumlah) DESC"

        rows = database.fetch_query(sql, params=tuple(params) if params else None, fetch_all=True)
        if rows:
            for row in rows:
                kategori = row['kategori'] if row['kategori'] else "Lainnya"
                hasil[kategori] = float(row[1]) if row[1] is not None else 0.0
        return hasil

Writing manajer_anggaran.py


In [6]:
%%writefile main_app.py
# main_app.py
import streamlit as st
import datetime
import pandas as pd

def format_rp(angka):
    return f"Rp {angka or 0:,.0f}".replace(",", ".")

from model import Transaksi
from manajer_anggaran import AnggaranHarian
from konfigurasi import KATEGORI_PENGELUARAN

st.set_page_config(page_title="Catatan Pengeluaran", layout="wide")

@st.cache_resource
def get_anggaran_manager():
    return AnggaranHarian()

anggaran = get_anggaran_manager()

def halaman_input(anggaran: AnggaranHarian):
    st.header("Tambah Pengeluaran Baru")
    with st.form("form_transaksi_baru", clear_on_submit=True):
        col1, col2 = st.columns([3, 1])
        with col1: deskripsi = st.text_input("Deskripsi*", placeholder="Contoh: Makan siang")
        with col2: kategori = st.selectbox("Kategori*:", KATEGORI_PENGELUARAN, index=0)
        col3, col4 = st.columns([1, 1])
        with col3: jumlah = st.number_input("Jumlah (Rp)*:", min_value=0.01, step=1000.0, format="%.0f", value=None)
        with col4: tanggal = st.date_input("Tanggal*:", value=datetime.date.today())

        if st.form_submit_button("Simpan Transaksi"):
            if not deskripsi or jumlah is None or jumlah <= 0:
                st.warning("Data input wajib diisi dengan benar!")
            else:
                tx = Transaksi(deskripsi, float(jumlah), kategori, tanggal)
                if anggaran.tambah_transaksi(tx):
                    st.success("Transaksi Berhasil Disimpan!")
                    st.cache_data.clear()
                    st.rerun()

def halaman_riwayat(anggaran: AnggaranHarian):
    st.header("Detail Semua Transaksi")
    if st.button("Refresh Riwayat"):
        st.cache_data.clear()
        st.rerun()
    df_transaksi = anggaran.get_dataframe_transaksi()
    if df_transaksi is None or df_transaksi.empty:
        st.info("Belum ada data transaksi.")
    else:
        st.dataframe(df_transaksi, use_container_width=True, hide_index=True)

def halaman_ringkasan(anggaran: AnggaranHarian):
    st.header("Ringkasan Pengeluaran")
    pilihan_periode = st.selectbox("Filter Periode:", ["Semua Waktu", "Hari Ini"])
    tanggal_filter = datetime.date.today() if pilihan_periode == "Hari Ini" else None

    total_pengeluaran = anggaran.hitung_total_pengeluaran(tanggal_filter)
    st.metric(label="Total Pengeluaran", value=format_rp(total_pengeluaran))

    st.divider()
    dict_per_kategori = anggaran.get_pengeluaran_per_kategori(tanggal_filter)
    if not dict_per_kategori:
        st.info("Tidak ada data pengeluaran.")
    else:
        data_kategori = [{"Kategori": kat, "Total": jml} for kat, jml in dict_per_kategori.items()]
        df_kategori = pd.DataFrame(data_kategori)
        st.bar_chart(df_kategori.set_index('Kategori')['Total'])

def main():
    st.sidebar.title("Navigasi")
    menu_pilihan = st.sidebar.radio("Pilih Menu:", ["Tambah", "Riwayat", "Ringkasan"])
    manajer = get_anggaran_manager()
    if menu_pilihan == "Tambah": halaman_input(manajer)
    elif menu_pilihan == "Riwayat": halaman_riwayat(manajer)
    elif menu_pilihan == "Ringkasan": halaman_ringkasan(manajer)

if __name__ == "__main__":
    main()

Writing main_app.py


In [8]:
# A. Ambil IP Publik untuk password LocalTunnel nanti
print("PASSWORD ENDPOINT IP ANDA:")
!curl ipv4.icanhazip.com

# B. Jalankan Server Streamlit untuk file utama praktikum dasar
!streamlit run main_app.py &>/dev/null &

# C. Buka Tunneling jaringan internet luar
!lt --port 8501

PASSWORD ENDPOINT IP ANDA:
34.61.123.21
your url is: https://fine-teams-fold.loca.lt
^C


***PENUGASAN***

In [9]:
%%writefile manajer_anggaran.py
# manajer_anggaran.py (VERSI PENUGASAN)
import datetime
import pandas as pd
from model import Transaksi
import database

class AnggaranHarian:
    _db_setup_done = False

    def __init__(self):
        if not AnggaranHarian._db_setup_done:
            if database.setup_database_initial():
                AnggaranHarian._db_setup_done = True

    def tambah_transaksi(self, transaksi: Transaksi) -> bool:
        if not isinstance(transaksi, Transaksi) or transaksi.jumlah <= 0: return False
        sql = "INSERT INTO transaksi (deskripsi, jumlah, kategori, tanggal) VALUES (?, ?, ?, ?)"
        params = (transaksi.deskripsi, transaksi.jumlah, transaksi.kategori, transaksi.tanggal.strftime("%Y-%m-%d"))
        last_id = database.execute_query(sql, params)
        if last_id is not None:
            transaksi.id = last_id
            return True
        return False

    # ===== FITUR PENUGASAN: METODE HAPUS TRANSAKSI =====
    def hapus_transaksi(self, id_transaksi: int) -> bool:
        """Menghapus data transaksi dari SQLite berdasarkan ID yang dipilih[cite: 500, 502]."""
        sql = "DELETE FROM transaksi WHERE id = ?"
        params = (id_transaksi,)
        result = database.execute_query(sql, params)
        return result is not None

    def get_dataframe_transaksi(self, filter_tanggal: datetime.date | None = None) -> pd.DataFrame:
        # Menampilkan kolom ID untuk mempermudah target penapisan hapus di UI [cite: 506]
        query = "SELECT id, tanggal, kategori, deskripsi, jumlah FROM transaksi"
        params = None
        if filter_tanggal:
            query += " WHERE tanggal = ?"
            params = (filter_tanggal.strftime("%Y-%m-%d"),)
        query += " ORDER BY tanggal DESC, id DESC"

        df = database.get_dataframe(query, params=params)
        if not df.empty:
            df.columns = ['ID', 'Tanggal', 'Kategori', 'Deskripsi', 'jumlah_raw']
            df['Jumlah (Rp)'] = df['jumlah_raw'].map(lambda x: f"Rp {x or 0:,.0f}".replace(",", "."))
            return df[['ID', 'Tanggal', 'Kategori', 'Deskripsi', 'Jumlah (Rp)']]
        return df

    def hitung_total_pengeluaran(self, tanggal: datetime.date | None = None) -> float:
        sql = "SELECT SUM(jumlah) FROM transaksi"
        params = None
        if tanggal:
            sql += " WHERE tanggal = ?"
            params = (tanggal.strftime("%Y-%m-%d"),)
        result = database.fetch_query(sql, params=params, fetch_all=False)
        if result and result[0] is not None: return float(result[0])
        return 0.0

    def get_pengeluaran_per_kategori(self, tanggal: datetime.date | None = None) -> dict:
        hasil = {}
        sql = "SELECT kategori, SUM(jumlah) FROM transaksi"
        params = []
        if tanggal:
            sql += " WHERE tanggal = ?"
            params.append(tanggal.strftime("%Y-%m-%d"))
        sql += " GROUP BY kategori HAVING SUM(jumlah) > 0 ORDER BY SUM(jumlah) DESC"

        rows = database.fetch_query(sql, params=tuple(params) if params else None, fetch_all=True)
        if rows:
            for row in rows:
                kategori = row['kategori'] if row['kategori'] else "Lainnya"
                hasil[kategori] = float(row[1]) if row[1] is not None else 0.0
        return hasil

Overwriting manajer_anggaran.py


In [10]:
%%writefile main_app.py
# main_app.py (VERSI PENUGASAN)
import streamlit as st
import datetime
import pandas as pd

def format_rp(angka):
    return f"Rp {angka or 0:,.0f}".replace(",", ".")

from model import Transaksi
from manajer_anggaran import AnggaranHarian
from konfigurasi import KATEGORI_PENGELUARAN

st.set_page_config(page_title="Catatan Pengeluaran + Hapus Fitur", layout="wide")

@st.cache_resource
def get_anggaran_manager():
    return AnggaranHarian()

anggaran = get_anggaran_manager()

def halaman_input(anggaran: AnggaranHarian):
    st.header("Tambah Pengeluaran Baru")
    with st.form("form_transaksi_baru", clear_on_submit=True):
        col1, col2 = st.columns([3, 1])
        with col1: deskripsi = st.text_input("Deskripsi*", placeholder="Contoh: Makan siang")
        with col2: kategori = st.selectbox("Kategori*:", KATEGORI_PENGELUARAN, index=0)
        col3, col4 = st.columns([1, 1])
        with col3: jumlah = st.number_input("Jumlah (Rp)*:", min_value=0.01, step=1000.0, format="%.0f", value=None)
        with col4: tanggal = st.date_input("Tanggal*:", value=datetime.date.today())

        if st.form_submit_button("Simpan Transaksi"):
            if not deskripsi or jumlah is None or jumlah <= 0:
                st.warning("Data input wajib diisi!")
            else:
                tx = Transaksi(deskripsi, float(jumlah), kategori, tanggal)
                if anggaran.tambah_transaksi(tx):
                    st.success("Berhasil disimpan!")
                    st.cache_data.clear()
                    st.rerun()

# ===== MODIFIKASI TAMPILAN UNTUK PENUGASAN HAPUS DATA =====
def halaman_riwayat(anggaran: AnggaranHarian):
    st.header("Detail Semua Transaksi")
    if st.button("Refresh Riwayat"):
        st.cache_data.clear()
        st.rerun()

    df_transaksi = anggaran.get_dataframe_transaksi()
    if df_transaksi is None or df_transaksi.empty:
        st.info("Belum ada data transaksi harian.")
    else:
        # Menampilkan tabel data yang memuat kolom 'ID' [cite: 506]
        st.dataframe(df_transaksi, use_container_width=True, hide_index=True)
        st.divider()

        # Penambahan Input ID & Tombol Eksekusi Hapus [cite: 509]
        st.subheader("Fungsionalitas Hapus Transaksi (Penugasan)")
        col_del1, col_del2 = st.columns([1, 2])
        with col_del1:
            id_target = st.number_input("ID Transaksi Hapus:", min_value=1, step=1, value=1)
        with col_del2:
            st.write("")
            st.write("")
            btn_hapus = st.button("Hapus Transaksi Terpilih")

        if btn_hapus:
            # Validasi apakah ID tersedia di dalam tabel data saat ini
            if id_target in df_transaksi['ID'].values:
                # Membuat Prompt Dialog Peringatan Konfirmasi [cite: 511]
                st.warning(f"Apakah Anda yakin ingin menghapus Transaksi dengan ID {id_target}?")
                if st.button("Konfirmasi Hapus Data"):
                    if anggaran.hapus_transaksi(int(id_target)):
                        st.success(f"Transaksi ID {id_target} sukses dihapus dari database! [cite: 515]")
                        st.cache_data.clear()  # Membersihkan cache data lama [cite: 516]
                        st.rerun()            # Memaksa aplikasi menggambar ulang layar [cite: 516]
                    else:
                        st.error("Gagal melakukan proses penghapusan data SQLite. [cite: 515]")
            else:
                st.error("ID Transaksi tidak ditemukan di dalam records!")

def halaman_ringkasan(anggaran: AnggaranHarian):
    st.header("Ringkasan Pengeluaran")
    pilihan_periode = st.selectbox("Filter Periode:", ["Semua Waktu", "Hari Ini"])
    tanggal_filter = datetime.date.today() if pilihan_periode == "Hari Ini" else None

    total_pengeluaran = anggaran.hitung_total_pengeluaran(tanggal_filter)
    st.metric(label="Total Pengeluaran", value=format_rp(total_pengeluaran))

    st.divider()
    dict_per_kategori = anggaran.get_pengeluaran_per_kategori(tanggal_filter)
    if not dict_per_kategori:
        st.info("Tidak ada data.")
    else:
        data_kategori = [{"Kategori": kat, "Total": jml} for kat, jml in dict_per_kategori.items()]
        df_kategori = pd.DataFrame(data_kategori)
        st.bar_chart(df_kategori.set_index('Kategori')['Total'])

def main():
    st.sidebar.title("Navigasi Aplikasi")
    menu_pilihan = st.sidebar.radio("Pilih Menu:", ["Tambah", "Riwayat", "Ringkasan"])
    manajer = get_anggaran_manager()
    if menu_pilihan == "Tambah": halaman_input(manajer)
    elif menu_pilihan == "Riwayat": halaman_riwayat(manajer)
    elif menu_pilihan == "Ringkasan": halaman_ringkasan(manajer)

if __name__ == "__main__":
    main()

Overwriting main_app.py


In [11]:
# A. Ambil IP Publik kembali
print("PASSWORD ENDPOINT IP ANDA:")
!curl ipv4.icanhazip.com

# B. Matikan proses port lama jika masih menyala kemudian hidupkan server baru
!fuser -k 8501/tcp
!streamlit run main_app.py &>/dev/null &

# C. Buka gerbang akses LocalTunnel baru
!lt --port 8501

PASSWORD ENDPOINT IP ANDA:
34.61.123.21
your url is: https://three-bottles-double.loca.lt
^C
